In [5]:
# ============================================================
# CELL 1: Imports + Modell laden
# ============================================================
import os
import shutil
import torch
import pandas as pd
from tqdm import tqdm
from PIL import Image, UnidentifiedImageError
import mobileclip

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Gerät: {device}")

weights_path = r"C:\Users\Lukas\masterthesis\thesis_2026\data\weights\mobileclip_weights\mobileclip_s1.pt"

model, _, preprocess = mobileclip.create_model_and_transforms(
    'mobileclip_s1',
    pretrained=weights_path,
    device=device
)
model.eval()
tokenizer = mobileclip.get_tokenizer('mobileclip_s1')
print("Modell geladen: MobileCLIP-S1")


# ============================================================
# CELL 2: Bereits ermitteltes Mixed-Prompt-Set (Simple für beide Klassen)
# ============================================================
categories = ["indoor", "outdoor"]

mixed_prompts = {
    "indoor":  "a photo of an indoor place",
    "outdoor": "a photo of an outdoor place"
}

category_texts = [mixed_prompts[cat] for cat in categories]
print("Verwendetes Prompt-Set:", mixed_prompts)


# ============================================================
# CELL 3: Klassifikationsfunktion
# ============================================================
def safe_open(path):
    try:
        return Image.open(path).convert("RGB")
    except (UnidentifiedImageError, OSError) as e:
        print(f"  ⚠ Skipped ({os.path.basename(path)}): {e}")
        return None

BATCH_SIZE = 32
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def classify(image_paths, category_texts):
    text_tokens = tokenizer(category_texts).to(device)
    with torch.no_grad():
        text_features = model.encode_text(text_tokens)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    results, confidences, valid_paths = [], [], []

    for i in tqdm(range(0, len(image_paths), BATCH_SIZE), desc="Batches"):
        batch_paths = image_paths[i : i + BATCH_SIZE]
        loaded = [(p, safe_open(p)) for p in batch_paths]
        loaded = [(p, img) for p, img in loaded if img is not None]
        if not loaded:
            continue

        paths_ok, imgs_ok = zip(*loaded)
        img_tensors = torch.stack([preprocess(img) for img in imgs_ok]).to(device)

        with torch.no_grad():
            image_features = model.encode_image(img_tensors)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)

        similarities = image_features @ text_features.T
        batch_preds  = similarities.argmax(dim=1).cpu().numpy()
        batch_conf   = similarities.max(dim=1).values.cpu().float().numpy()

        results.extend([categories[idx] for idx in batch_preds])
        confidences.extend(batch_conf)
        valid_paths.extend(paths_ok)

    return valid_paths, results, confidences


# ============================================================
# CELL 4: FINALE Klassifikation — dein gesamter Datensatz
# ============================================================
full_dataset_folder = r"C:\Users\Lukas\masterthesis\thesis_2026\data\Annotation_Indoor_Outdoor\random_sample_images_indoor_outdoor"
output_csv = r"C:\Users\Lukas\masterthesis\thesis_2026\data\Annotation_Indoor_Outdoor\random_sample_indoor_outdoor_classification_simple_prompts.csv"

full_image_paths = []
for root, _, files in os.walk(full_dataset_folder):
    for fname in sorted(files):
        ext = os.path.splitext(fname)[1].lower()
        if ext in VALID_EXTENSIONS:
            full_image_paths.append(os.path.join(root, fname))

print(f"Bilder im Gesamtdatensatz gefunden: {len(full_image_paths)}")

valid_paths, results, confidences = classify(full_image_paths, category_texts)

df_full_results = pd.DataFrame({
    "file_name": [os.path.basename(p) for p in valid_paths],
    "file_path": valid_paths,
    "pred_label": results,
    "confidence": confidences
})
df_full_results.to_csv(output_csv, index=False)

print(f"\n✅ Klassifikation abgeschlossen.")
print(f"Gespeichert: {output_csv}")
print("\nVerteilung im Gesamtdatensatz:")
print(df_full_results["pred_label"].value_counts())
print(f"\nGesamt klassifiziert: {len(df_full_results)} Bilder")


# ============================================================
# CELL 5: Bilder nach Vorhersage in Ordner sortieren (Kopieren)
# ============================================================
output_base = r"C:\Users\Lukas\masterthesis\thesis_2026\data\Annotation_Indoor_Outdoor\random_sample_indoor_outdoor_classification_CLIP_simple_prompts_images_sorted"

sorted_dirs = {cat: os.path.join(output_base, cat) for cat in categories}
for d in sorted_dirs.values():
    os.makedirs(d, exist_ok=True)

for path, pred in tqdm(zip(valid_paths, results), total=len(valid_paths), desc="Kopiere Bilder"):
    dest = os.path.join(sorted_dirs[pred], os.path.basename(path))
    if not os.path.exists(dest):
        shutil.copy(path, dest)

print("\nBilder sortiert:")
for cat in categories:
    print(f"  {cat}: {results.count(cat)} → {sorted_dirs[cat]}")

Gerät: cuda
Modell geladen: MobileCLIP-S1
Verwendetes Prompt-Set: {'indoor': 'a photo of an indoor place', 'outdoor': 'a photo of an outdoor place'}
Bilder im Gesamtdatensatz gefunden: 300


Batches: 100%|█████████████████████████████████████████████████████████████████████████| 10/10 [00:17<00:00,  1.76s/it]



✅ Klassifikation abgeschlossen.
Gespeichert: C:\Users\Lukas\masterthesis\thesis_2026\data\Annotation_Indoor_Outdoor\random_sample_indoor_outdoor_classification_simple_prompts.csv

Verteilung im Gesamtdatensatz:
pred_label
outdoor    286
indoor      14
Name: count, dtype: int64

Gesamt klassifiziert: 300 Bilder


Kopiere Bilder: 100%|███████████████████████████████████████████████████████████████| 300/300 [00:00<00:00, 604.70it/s]


Bilder sortiert:
  indoor: 14 → C:\Users\Lukas\masterthesis\thesis_2026\data\Annotation_Indoor_Outdoor\random_sample_indoor_outdoor_classification_CLIP_simple_prompts_images_sorted\indoor
  outdoor: 286 → C:\Users\Lukas\masterthesis\thesis_2026\data\Annotation_Indoor_Outdoor\random_sample_indoor_outdoor_classification_CLIP_simple_prompts_images_sorted\outdoor
